# Tutorial 2 — Materials and phantoms

**Goal:** understand how DIANA describes matter, and build your own phantoms — from
geometric primitives or from a real segmented volume.

**You will learn:** `MATERIALS`, `material_from_formula`, `register_material`,
`PhantomBuilder`, preset phantoms, and `phantom_from_array`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import neutron_xray_sim as nxs

print("DIANA / neutron_xray_sim", nxs.__version__)

## 1. Materials

A `Material` stores its density, its thermal-neutron attenuation (split into absorption,
coherent and incoherent scattering) and its X-ray attenuation on a 13-point energy grid.
All attenuation coefficients are **linear**, in cm⁻¹.

In [ ]:
print(len(nxs.MATERIALS), "built-in materials:\n", sorted(nxs.MATERIALS))
print()
for key in ["water", "hdpe", "aluminum", "iron", "titanium"]:
    print(nxs.MATERIALS[key])

The complementarity is easy to see if we plot every material in the $(\mu_x, \mu_n)$ plane
at 80 keV: metals spread along $\mu_x$, hydrogenous materials along $\mu_n$.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for key, m in nxs.MATERIALS.items():
    if key == "air" or m.mu_x_at(80) > 8:
        continue
    ax.scatter(m.mu_x_at(80), m.mu_n, color=m.color, edgecolor="k", s=60)
    ax.annotate(m.symbol, (m.mu_x_at(80), m.mu_n), xytext=(4, 3),
                textcoords="offset points", fontsize=8)
ax.set_xlabel(r"$\mu_x$ at 80 keV [cm$^{-1}$]"); ax.set_ylabel(r"$\mu_n$ [cm$^{-1}$]")
ax.set_title("Where materials sit in the bimodal plane"); ax.grid(alpha=.3)

X-ray attenuation depends strongly on energy (note the K-edge of lead near 88 keV):

In [ ]:
E = np.linspace(20, 300, 300)
fig, ax = plt.subplots(figsize=(6, 4))
for key in ["water", "aluminum", "titanium", "iron", "lead"]:
    ax.semilogy(E, nxs.MATERIALS[key].mu_x_array(E), label=key)
ax.set_xlabel("photon energy [keV]"); ax.set_ylabel(r"$\mu_x$ [cm$^{-1}$]"); ax.legend()

### New materials from a chemical formula

`material_from_formula` computes both modalities from the elemental composition (NIST XCOM
data for X-rays, bound-atom cross sections for neutrons). Hydrogenous materials often need
`incoherent_scale < 1`, because the bound-atom incoherent cross section of hydrogen
overestimates the attenuation seen in imaging.

`register_material` adds it to the database so it can be referred to by name everywhere.

In [ ]:
pmma = nxs.material_from_formula(
    name="PMMA (acrylic)", symbol="PMMA", formula="C5H8O2",
    density_gcc=1.18, color="#AADDFF", incoherent_scale=0.5,
)
nxs.register_material("pmma", pmma, overwrite=True)
print(pmma)

Mixtures of phases (electrodes, rocks, fossil bone…) use `make_composite_material` with
`(formula, weight_fraction, end-member density)` tuples. Here: a porous LFP cathode coating
(active material + carbon black + PVDF binder).

In [ ]:
cathode = nxs.make_composite_material(
    "LFP cathode coating", "LFPc", bulk_density_gcc=2.2,
    components=[("LiFePO4", 0.90, 3.6), ("C", 0.05, 2.0), ("C2H2F2", 0.05, 1.78)],
)
print(cathode)

> **Note:** building a material needs X-ray data for each element. The repository ships
> NIST XCOM files for H, Li, C, O, F, P, Mn, Fe, Co and Ni; for any other element you get an
> error naming the file to add (see `docs/installation.md`).

## 2. Building a phantom

A phantom is a **label volume** (one integer per voxel, indexing a list of materials) plus the
attenuation volumes derived from it. `PhantomBuilder` composites primitives; later calls
overwrite earlier ones, so build from the outside in.

**Conventions:** arrays are stored as `(Nz, Nx, Ny)`; coordinates and centres are `(z, x, y)`
in cm, measured from the volume centre. Label 0 is always air.

In [ ]:
b = nxs.PhantomBuilder(N=64, voxel_cm=0.02)          # 64³ voxels, 1.28 cm cube
b.add_cylinder("aluminum", radius_cm=0.55)             # full-height Al cylinder (axis z)
b.add_cylinder("pmma", radius_cm=0.50)                 # acrylic core
b.add_sphere("water", center_cm=(0.0, 0.2, 0.15), radius_cm=0.12)
b.add_rod("iron", center_cm=(-0.2, -0.1), radius_cm=0.07)          # rod: centre is (x, y)
b.add_box("titanium", center_cm=(0.2, -0.2, 0.2), half_extents_cm=(0.1, 0.06, 0.06))
b.add_hollow_cylinder("hdpe", inner_radius_cm=0.3, outer_radius_cm=0.34, height_cm=0.4)

# Anything the primitives cannot express: paint a boolean mask on the coordinate grids.
b.paint("air", (b.Z > 0.45) & (b.X**2 + b.Y**2 < 0.2**2))   # a drilled hole at the top

my_phantom = b.build("tutorial_phantom")
print(my_phantom)

In [ ]:
def show_labels(ph, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    mid = [s // 2 for s in ph.shape]
    views = [ph.label_vol[mid[0]], ph.label_vol[:, mid[1], :], ph.label_vol[:, :, mid[2]]]
    for ax, v, t in zip(axes, views, ["axial (z = const)", "x = const", "y = const"]):
        ax.imshow(v, cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
        ax.set_title(t); ax.axis("off")
    fig.suptitle(title or ph.name)

show_labels(my_phantom)
print({i: m.name for i, m in enumerate(my_phantom.materials)})

## 3. Preset phantoms

Presets are ready-made samples at physically realistic sizes. Voxel size scales with `N`, so
changing the resolution never changes the geometry.

In [ ]:
print(list(nxs.PHANTOM_PRESETS))
for preset in ["battery", "bone_implant", "industrial"]:
    show_labels(nxs.make_phantom(preset, N=48), preset)

Presets accept extra options. Non-cubic grids use `Nx, Ny, Nz`:

In [ ]:
tall = nxs.make_phantom("composite", Nx=48, Ny=48, Nz=24)
print(tall.shape)
roll = nxs.make_phantom("jellyroll_battery", N=96, n_jellyroll_turns=1)
show_labels(roll, "jellyroll battery (1 turn)")

## 4. Importing a real segmented volume

If you have a segmentation of a real sample (a labelled `.npy`, `.npz`, `.tif` stack, or an
array in memory), map its labels to material names and you get a normal phantom — ready for
the same simulation pipeline. Labels need not be contiguous.

In [ ]:
seg = np.zeros((32, 64, 64), dtype=np.uint16)
yy, xx = np.mgrid[:64, :64]
seg[:, (yy - 32)**2 + (xx - 32)**2 < 28**2] = 10     # matrix
seg[:, (yy - 25)**2 + (xx - 40)**2 < 6**2] = 42      # inclusion

imported = nxs.phantom_from_array(
    seg, {"name": "my_scan", "voxel_cm": 0.02,
          "class_map": {0: "air", 10: "hdpe", 42: "steel"}},
)
print(imported)
show_labels(imported)

For files use `nxs.phantom_from_segmented_volume("scan.tif", "metadata.json")`; see
`docs/importing-data.md` for the metadata format.

**Exercise:** design a phantom of something you work with. Can you find two materials that
overlap in $\mu_x$ but separate in $\mu_n$?